# 18 - Context Management (Summarization)

## Purpose

This notebook demonstrates a LangChain DeepAgents tutorial episode. It walks through the key concepts and code patterns for building, configuring, and operating AI agents with the `deepagents` framework.

## What You'll Learn

- How deep agents differ from regular LangChain agents
- The built-in tool suite every agent receives automatically
- How to configure models, backends, permissions, and memory
- Practical patterns for tools, planning, memory, and handoffs

## Tutorial Series Context

This is Episode 18-context_management of the LangChain DeepAgents YouTube tutorial series. Each episode focuses on a specific aspect of agent development, from basic introduction to advanced patterns like subagents, memory, permissions, and skills.

## How to Use This Notebook

1. **Run cells top-to-bottom**: The code is designed to execute in order, with each section building on the previous one.
2. **Observe tool calls**: Watch for `tool_calls` in message outputs - this shows which agents/tools are being used.
3. **Modify and experiment**: The code is intentionally kept identical to the original Python scripts. Feel free to modify parameters, models, or prompts to see how behavior changes.
4. **Refer to comments**: Each code section has accompanying markdown explanations of what the code does and why.

## Key Concepts Covered in This Episode

- SummarizationMiddleware compresses old conversation turns
- Auto-trigger near model context limit (~85%) impossible to demo cheaply
- Supply custom trigger: when (messages, 4) -> once history exceeds 4
- keep = what to keep verbatim (messages, 2) -> 2 newest messages
- History compression: summary standing in for early turns + recent kept verbatim
- Automatic tool result offloading >20k tokens to file with preview + path
- Details from summarized turns should still be answerable - that's the whole point


# Overview: 18-context_management.ipynb

This notebook converts the Python tutorial `18-context_management.ipynb` into an interactive Jupyter Notebook format, preserving all code examples and explanations exactly as they appear in the original tutorial scripts.

## Notebook Structure

Each logical section of the original Python script has been split into its own code cell. Each code cell is preceded by a markdown cell containing:

- **A short heading** describing what that section does
- **1–3 sentences** explaining the purpose and key concepts of that section

## How to Navigate

- **Run cells in order**: The tutorial was designed to be executed top-to-bottom. Each section builds on the previous one.
- **Observe tool calls**: Watch the output for `tool_calls` entries - this shows which built-in or tools the agent invokes.
- **Read the markdown comments**: Each code section has a preceding explanation of what the code does and why it's structured that way.
- **Experiment freely**: The original `.py` files are unchanged. Feel free to modify, extend, or experiment with the code in this notebook.

## Educational Value

This notebook serves as both a working tutorial replication and a learning resource. By running each cell and reading the accompanying explanations, you'll gain hands-on experience with:

- LangChain DeepAgents framework architecture
- Backend configuration (virtual vs real filesystem)
- Tool definition and registration
- Planning, memory, permissions, and context management
- Subagent delegation and structured output
- Streaming modes and human-in-the-loop patterns

## Tips for Getting the Most Out of This Notebook

1. **Don't skip the markdown cells**: The educational content between code sections is essential for understanding the "why" behind the code.
2. **Re-run after modifications**: After experimenting with changes, re-run all cells from the top to ensure everything still works together.
3. **Refer to the original Python scripts**: The `.py` files in the same directory are unchanged and can be referenced for the exact original code.
4. **Use rich output**: This notebook uses Rich formatting for colorful console output - pay attention to color-coded hints and status messages.

---

_Notebook automatically generated from `18-context_management.ipynb`. Original Python scripts preserved unchanged._


## Section 2: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section sets up the imports and environment configurati
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
"""
18 - Context Management (Summarization)
=======================================
- Long conversations eventually exceed the model's context window
- `SummarizationMiddleware` compresses old turns into a summary automatically
- By default it fires near the context limit; here we force it early to watch it
Run:  python 18-context_management.py
"""

## Section 4: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 1: Imports and setup
import os

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from rich import print

load_dotenv()
MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")

## Section 6: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements force summarization to trigger early
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 2: Force summarization to trigger early
# Every deep agent already summarizes automatically, but only near the model's
# context limit (~85%) — impossible to demo cheaply. So we supply our own
# instance with a tiny trigger.
#   trigger = when to summarize      ("messages", 4) -> once history exceeds 4
#   keep    = what to keep verbatim  ("messages", 2) -> the 2 newest messages
#
# Because its `.name` matches the built-in ("SummarizationMiddleware"),
# deepagents REPLACES the default with ours instead of stacking a second one.
summarizer = SummarizationMiddleware(
    model=MODEL,
    trigger=("messages", 4),
    keep=("messages", 2),
)

agent = create_deep_agent(
    model=MODEL,
    middleware=[summarizer],
    checkpointer=InMemorySaver(),  # so all turns share one growing history
    system_prompt="You are a travel assistant. Reply in one short sentence.",
)

CONFIG = {"configurable": {"thread_id": "trip-planning"}}


def turn(text: str) -> int:
    """Send one message and return how many messages the history now holds."""
    result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config=CONFIG)
    return len(result["messages"])

## Section 8: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements have a conversation long enough to t
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 3: Have a conversation long enough to trip the trigger
# Each exchange adds ~2 messages, so history crosses the limit after a few turns
# and the middleware compresses the older ones.
conversation = [
    "I'm planning a trip to Japan in April.",
    "I want to see cherry blossoms.",
    "My budget is about 3000 dollars.",
    "I prefer trains over flying domestically.",
    "I'm travelling with my sister.",
]

print("[bold cyan]Message count after each turn:[/bold cyan]")
previous = 0
for text in conversation:
    count = turn(text)
    # Without summarization this would climb 2, 4, 6, 8, 10... Instead it
    # plateaus: old turns keep getting folded into a compact summary.
    marker = " [green]<- compressed[/green]" if count <= previous else ""
    print(f"  after '{text[:34]:34}' -> {count:2} messages{marker}")
    previous = count

## Section 10: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 4: Look at what survived
# The history now holds a summary standing in for the early turns, plus the
# most recent messages kept verbatim.
state = agent.get_state(CONFIG)
print(
    f"\n[bold cyan]Final history ({len(state.values['messages'])} messages, not {2 * len(conversation)}):[/bold cyan]"
)
for message in state.values["messages"]:
    kind = message.__class__.__name__.replace("Message", "")
    content = message.content
    if isinstance(content, list):
        content = "".join(b.get("text", "") for b in content if isinstance(b, dict))
    print(f"  [magenta]{kind:6}[/magenta] {str(content).strip()[:88]}")

## Section 12: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 5: Compression is not amnesia
# The details from the early, summarized turns should still be answerable —
# that is the whole point: fewer tokens, same knowledge.
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Remind me: when am I going, and with whom?"}]},
    config=CONFIG,
)
answer = result["messages"][-1].content
if isinstance(answer, list):
    answer = "".join(b.get("text", "") for b in answer if isinstance(b, dict))
print(f"\n[bold green]Recall after compression:[/bold green] {str(answer).strip()}")

## Section 14: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be This section explains context management including summarization and automatic tool result offloadin, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section explains context management including summariza
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 6: The other half of context management
# Besides summarizing conversations, deepagents automatically OFFLOADS huge tool
# results (>20k tokens) to a file in the backend and leaves a short preview plus
# the path behind. The agent can read_file it back on demand — so one giant API
# response can never blow up the whole conversation.
print("\n[dim]Also automatic: tool results over ~20k tokens are written to a[/dim]")
print("[dim]file and replaced with a preview + path the agent can read later.[/dim]")

# %%